In [0]:
import requests
import datetime
import json

In [0]:
df = spark.table('bronze.pokemon.pokemon').select('url').limit(1)
df.display()

In [0]:
spark.table('bronze.pokemon.pokemon').select('url').limit(1).collect()[0][0]

In [0]:
urls = spark.table('bronze.pokemon.pokemon').select('url').distinct().toPandas()['url'].toList()

urls

In [0]:
def save_pokemon(data):
    now = datetime.datetime.now().strftime('%Y-%m-%d_%H:%M:%S.%f')
    data['date_ingestion'] = datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    filename = f'/dbfs/mnt/datalake/pokemon/pokemon/details/{data['id']}_{now}.json'

    with open(filename, 'w') as open_file:
        json.dump(data, open_file)

def get_and_save(url):
    response = requests.get(url)
    
    if response.status_code == 200:
        data = response.json()
        save_pokemon(data)
    else:
        print(f'Error: {response.status_code}')
        print(f'URL: {url}')

In [0]:
from multiprocessing import Pool

with Pool(5) as p:
    print(p.map(get_and_save, urls))

get_and_save('https://pokeapi.co/api/v2/pokemon/1/')

In [0]:
df = spark.read.json('/mnt/datalake/pokemon/pokemon/details')
df.display()